# Light Curve Viewer
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
KUPT-lightcurveJSON (V2) viewer and period-analysis tool.<br>
<br>
**History**<br>
coding 2026-07-07 : 1st coding<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Parameters
**Input / Output files settings**

In [ ]:
from pathlib import Path

JSON_PATH = Path(
    "/Volumes/SSD-PEU4A/VScode/satphotometry_photometry/output/lightcurve/lightcurve_EKRAN_2_DEB_1977-092J_2025-10-10_095522_095722.json"
)

PERIOD_ANAYSIS = True

# Save figures in addition to displaying them.
SAVE_FIGURES = True
OUTPUT_DIRECTORY = Path("/Users/kiyoaki/VScode/satphotometry_photometry/output/lightcurve_analysis")

**Plot settings**

In [ ]:
# Quantity displayed on the light-curve and phase-folded plots.
# Available choices:
#   "mag_app"     : apparent magnitude
#   "mag_in"      : instrumental magnitude
#   "object_flux" : background-subtracted flux
PLOT_COLUMN = "mag_app"

# Quantity used for period estimation.
# Flux is generally preferable because it is a linear quantity.
PERIOD_ANALYSIS_COLUMN = "object_flux"

# Period-search range [seconds]
MIN_PERIOD = 0.5
MAX_PERIOD = 20

# Number of frequency samples used in the Lomb-Scargle search
N_FREQUENCIES = 100_000

# Set a numerical value to use a manually specified period.
# Set to None to use the Lomb-Scargle best period.
MANUAL_PERIOD = None
# Example:
# MANUAL_PERIOD = 10.023

# Reference time used to define phase zero [seconds].
# None means that the first valid observation is used.
PHASE_ZERO_TIME = None

# Plot two cycles in the phase-folded light curve.
PLOT_TWO_PHASE_CYCLES = True

# Number of bins used to calculate phase-binned averages.
# Set to None to disable phase binning.
N_PHASE_BINS = 50

# Save the phase-folded light curve as JSON.
SAVE_PHASE_FOLDED_JSON = True

# Save phase-binned values in addition to individual measurements.
SAVE_PHASE_BINNED_DATA = True

### Import and initial settings

In [ ]:
from __future__ import annotations

import json
from typing import Any

from copy import deepcopy
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
from astropy.timeseries import LombScargle

### JSON loading

In [ ]:
def load_kupt_lightcurve_json(json_path: Path) -> dict[str, Any]:
    """Load a KUPT-lightcurveJSON file."""

    if not json_path.exists():
        raise FileNotFoundError(
            f"KUPT-lightcurveJSON file was not found:\n{json_path.resolve()}"
        )

    with json_path.open("r", encoding="utf-8") as file:
        kupt_data = json.load(file)

    if "header" not in kupt_data:
        raise KeyError("The JSON file does not contain the 'header' key.")

    if "data" not in kupt_data:
        raise KeyError("The JSON file does not contain the 'data' key.")

    if not isinstance(kupt_data["data"], list):
        raise TypeError("The value of 'data' must be a list.")

    return kupt_data

### Helper functions

In [ ]:
def get_nested(
    dictionary: dict[str, Any],
    *keys: str,
    default: Any = "N/A",
) -> Any:
    """Safely obtain a value from nested dictionaries."""

    value: Any = dictionary

    for key in keys:
        if not isinstance(value, dict) or key not in value:
            return default

        value = value[key]

    if value is None:
        return default

    return value


def format_value(
    value: Any,
    unit: str = "",
    digits: int | None = None,
) -> str:
    """Format a metadata value for display."""

    if value == "N/A" or value is None:
        return "N/A"

    if digits is not None:
        try:
            text = f"{float(value):.{digits}f}"
        except (TypeError, ValueError):
            text = str(value)
    else:
        text = str(value)

    if unit:
        return f"{text} {unit}"

    return text


def print_section(title: str) -> None:
    """Print a formatted section heading."""

    print()
    print("=" * 72)
    print(title)
    print("=" * 72)

### Observation-information display

In [ ]:
def display_observation_information(kupt_data: dict[str, Any]) -> None:
    """Display target, observer, instrument, and photometry settings."""

    header = kupt_data["header"]

    object_info = header.get("object", {})
    capture = header.get("captureSettings", {})
    photometry = header.get("photParams", {})
    observer = header.get("observer", {})
    editor = header.get("editor", {})
    measure = header.get("measureSettings", {})
    aperture = measure.get("measureFieldSize", {})

    print_section("Observation target")

    print(
        f"{'Object name':28s}: "
        f"{get_nested(object_info, 'objectName')}"
    )
    print(
        f"{'International designator':28s}: "
        f"{get_nested(object_info, 'intlDES')}"
    )
    print(
        f"{'NORAD catalog ID':28s}: "
        f"{get_nested(object_info, 'noradID')}"
    )
    print(
        f"{'Start time UTC':28s}: "
        f"{get_nested(header, 'startDateTime')}"
    )
    print(
        f"{'End time UTC':28s}: "
        f"{get_nested(header, 'endDateTime')}"
    )
    print(
        f"{'Number of frames':28s}: "
        f"{get_nested(header, 'length', default=len(kupt_data['data']))}"
    )

    print_section("Observer and observatory")

    print(
        f"{'Observer':28s}: "
        f"{get_nested(observer, 'observerName')}"
    )
    print(
        f"{'Observatory':28s}: "
        f"{get_nested(observer, 'observatory')}"
    )

    print_section("Capture settings")

    print(
        f"{'Image size':28s}: "
        f"{get_nested(capture, 'Xnaxis')} × "
        f"{get_nested(capture, 'Ynaxis')} pixel"
    )
    print(
        f"{'Binning':28s}: "
        f"{get_nested(capture, 'Xbinning')} × "
        f"{get_nested(capture, 'Ybinning')}"
    )
    print(
        f"{'Exposure time':28s}: "
        f"{format_value(get_nested(capture, 'exposure'), 's', 7)}"
    )
    print(
        f"{'Gain':28s}: "
        f"{format_value(get_nested(capture, 'gain'), digits=3)}"
    )
    print(
        f"{'Offset':28s}: "
        f"{format_value(get_nested(capture, 'offset'), digits=3)}"
    )
    print(
        f"{'Camera ID':28s}: "
        f"{get_nested(capture, 'camID')}"
    )
    print(
        f"{'Capture software':28s}: "
        f"{get_nested(capture, 'swcreate')}"
    )

    print_section("Photometry settings")

    print(
        f"{'Zeropoint':28s}: "
        f"{format_value(get_nested(photometry, 'ZP'), 'mag', 6)}"
    )
    print(
        f"{'Exposure-corrected ZP':28s}: "
        f"{format_value(get_nested(photometry, 'ZP_exp'), 'mag', 6)}"
    )
    print(
        f"{'Zeropoint scatter':28s}: "
        f"{format_value(get_nested(photometry, 'ZP_scatter'), 'mag', 6)}"
    )
    print(
        f"{'Selected zeropoint mode':28s}: "
        f"{get_nested(photometry, 'selected_mode')}"
    )
    print(
        f"{'Pixel area':28s}: "
        f"{format_value(get_nested(photometry, 'pixel_area_arcsec2'), 'arcsec²', 6)}"
    )
    print(
        f"{'Aperture area':28s}: "
        f"{format_value(get_nested(photometry, 'aperture_area_arcsec2'), 'arcsec²', 6)}"
    )

    print(
        f"{'Aperture shape':28s}: "
        f"{get_nested(measure, 'measureFieldShape')}"
    )
    print(
        f"{'Aperture radius':28s}: "
        f"{format_value(get_nested(aperture, 'aperture_radius_pixel'), 'pixel', 3)}"
    )
    print(
        f"{'Background annulus inner':28s}: "
        f"{format_value(get_nested(aperture, 'annulus_r_in_pixel'), 'pixel', 3)}"
    )
    print(
        f"{'Background annulus outer':28s}: "
        f"{format_value(get_nested(aperture, 'annulus_r_out_pixel'), 'pixel', 3)}"
    )

    print_section("Light-curve creation software")

    print(
        f"{'Application':28s}: "
        f"{get_nested(editor, 'appName')}"
    )
    print(
        f"{'Version':28s}: "
        f"{get_nested(editor, 'version')}"
    )

### Light curve

In [ ]:
# ============================================================
# Light-curve extraction
# ============================================================

def extract_lightcurve(
    kupt_data: dict[str, Any],
    value_column: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract valid light-curve measurements.

    Returns
    -------
    elapsed_seconds
        Elapsed time from the beginning of the observation.
    values
        Values of the selected measurement column.
    original_ids
        Original frame IDs in the JSON file.
    """

    valid_columns = {
        "mag_app",
        "mag_in",
        "object_flux",
        "aperture_sum",
        "bkg_med_pix",
        "x_centroid",
        "y_centroid",
    }

    if value_column not in valid_columns:
        raise ValueError(
            f"Unsupported column: {value_column}\n"
            f"Available columns: {sorted(valid_columns)}"
        )

    elapsed_seconds: list[float] = []
    values: list[float] = []
    original_ids: list[int] = []

    for row in kupt_data["data"]:
        time_value = row.get("elapsed_seconds")
        measurement = row.get(value_column)

        # Reject missing values such as JSON null.
        if time_value is None or measurement is None:
            continue

        try:
            time_float = float(time_value)
            measurement_float = float(measurement)
        except (TypeError, ValueError):
            continue

        if not np.isfinite(time_float) or not np.isfinite(measurement_float):
            continue

        # Magnitudes and object flux are valid only for successful photometry.
        if value_column in {"mag_app", "mag_in", "object_flux"}:
            if row.get("status") != "Success":
                continue

        # A non-positive object flux is unsuitable for period analysis.
        if value_column == "object_flux" and measurement_float <= 0:
            continue

        elapsed_seconds.append(time_float)
        values.append(measurement_float)
        original_ids.append(int(row.get("id", len(original_ids))))

    if len(elapsed_seconds) == 0:
        raise ValueError(
            f"No valid measurements were found for '{value_column}'."
        )

    time_array = np.asarray(elapsed_seconds, dtype=float)
    value_array = np.asarray(values, dtype=float)
    id_array = np.asarray(original_ids, dtype=int)

    sort_index = np.argsort(time_array)

    return (
        time_array[sort_index],
        value_array[sort_index],
        id_array[sort_index],
    )


def get_axis_label(column: str) -> str:
    """Return an appropriate axis label for a data column."""

    labels = {
        "mag_app": "Apparent magnitude",
        "mag_in": "Instrumental magnitude",
        "object_flux": "Object flux [ADU]",
        "aperture_sum": "Aperture sum [ADU]",
        "bkg_med_pix": "Median background [ADU pixel$^{-1}$]",
        "x_centroid": "X centroid [pixel]",
        "y_centroid": "Y centroid [pixel]",
    }

    return labels.get(column, column)


def is_magnitude_column(column: str) -> bool:
    """Return True when the selected quantity is a magnitude."""

    return column in {"mag_app", "mag_in"}


# ============================================================
# Light-curve plot
# ============================================================

def plot_lightcurve(
    time_seconds: np.ndarray,
    values: np.ndarray,
    column: str,
    object_name: str,
    intl_des: str,
    output_path: Path | None = None,
) -> None:
    """Plot the time-series light curve."""

    plt.figure(figsize=(12, 5))

    plt.scatter(
        time_seconds,
        values,
        s=10,
        alpha=0.8,
        label="Valid measurements",
    )

    plt.xlabel("Elapsed time [s]")
    plt.ylabel(get_axis_label(column))
    plt.title(f"{object_name} ({intl_des}) light curve")
    plt.grid(alpha=0.3)

    if is_magnitude_column(column):
        plt.gca().invert_yaxis()

    plt.legend()
    plt.tight_layout()

    if output_path is not None:
        plt.savefig(output_path, dpi=200, bbox_inches="tight")

    plt.show()


### Period estimation

In [ ]:
# ============================================================
# Lomb-Scargle period estimation
# ============================================================

def estimate_period_lomb_scargle(
    time_seconds: np.ndarray,
    values: np.ndarray,
    min_period: float,
    max_period: float,
    n_frequencies: int,
) -> tuple[float, np.ndarray, np.ndarray]:
    """
    Estimate the strongest period using a Lomb-Scargle periodogram.
    """

    if min_period <= 0:
        raise ValueError("MIN_PERIOD must be positive.")

    if max_period <= min_period:
        raise ValueError("MAX_PERIOD must be greater than MIN_PERIOD.")

    time_span = np.ptp(time_seconds)

    if max_period >= time_span:
        print(
            "Warning: MAX_PERIOD is comparable to or longer than the "
            "observation duration."
        )

    # Remove a constant offset and scale the values.
    centered_values = values - np.nanmedian(values)

    value_scale = np.nanstd(centered_values)

    if not np.isfinite(value_scale) or value_scale == 0:
        raise ValueError(
            "The selected period-analysis data have no measurable variation."
        )

    normalized_values = centered_values / value_scale

    minimum_frequency = 1.0 / max_period
    maximum_frequency = 1.0 / min_period

    frequencies = np.linspace(
        minimum_frequency,
        maximum_frequency,
        n_frequencies,
    )

    lomb_scargle = LombScargle(
        time_seconds,
        normalized_values,
        normalization="standard",
    )

    power = lomb_scargle.power(frequencies)

    best_index = int(np.nanargmax(power))
    best_frequency = frequencies[best_index]
    best_period = 1.0 / best_frequency

    return best_period, frequencies, power


def find_periodogram_peaks(
    frequencies: np.ndarray,
    power: np.ndarray,
    number_of_peaks: int = 10,
    minimum_separation_fraction: float = 0.01,
) -> list[tuple[float, float]]:
    """
    Find major periodogram peaks while avoiding nearly identical frequencies.
    """

    sorted_indices = np.argsort(power)[::-1]
    selected: list[tuple[float, float]] = []

    frequency_span = frequencies.max() - frequencies.min()
    minimum_separation = minimum_separation_fraction * frequency_span

    for index in sorted_indices:
        frequency = frequencies[index]
        period = 1.0 / frequency
        peak_power = power[index]

        is_separated = all(
            abs(frequency - selected_frequency) >= minimum_separation
            for selected_frequency, _ in selected
        )

        if is_separated:
            selected.append((frequency, peak_power))

        if len(selected) >= number_of_peaks:
            break

    return [
        (1.0 / frequency, peak_power)
        for frequency, peak_power in selected
    ]


def plot_periodogram(
    frequencies: np.ndarray,
    power: np.ndarray,
    best_period: float,
    object_name: str,
    intl_des: str,
    output_path: Path | None = None,
) -> None:
    """Plot the Lomb-Scargle periodogram as a function of period."""

    periods = 1.0 / frequencies

    sort_index = np.argsort(periods)
    periods = periods[sort_index]
    sorted_power = power[sort_index]

    plt.figure(figsize=(10, 5))

    plt.plot(periods, sorted_power, linewidth=1.0)

    plt.axvline(
        best_period,
        linestyle="--",
        linewidth=1.5,
        label=f"Best period = {best_period:.6f} s",
    )

    plt.xlabel("Period [s]")
    plt.ylabel("Lomb-Scargle power")
    plt.title(f"{object_name} ({intl_des}) Lomb-Scargle periodogram")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    if output_path is not None:
        plt.savefig(output_path, dpi=200, bbox_inches="tight")

    plt.show()


# ============================================================
# Phase-folded light curve
# ============================================================

def calculate_phase(
    time_seconds: np.ndarray,
    period: float,
    phase_zero_time: float | None = None,
) -> np.ndarray:
    """Calculate phase values between 0 and 1."""

    if period <= 0:
        raise ValueError("The folding period must be positive.")

    if phase_zero_time is None:
        phase_zero_time = float(np.min(time_seconds))

    return ((time_seconds - phase_zero_time) / period) % 1.0


def calculate_phase_binned_curve(
    phase: np.ndarray,
    values: np.ndarray,
    number_of_bins: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Calculate the median and standard deviation in equally spaced phase bins.
    """

    if number_of_bins < 2:
        raise ValueError("N_PHASE_BINS must be at least 2.")

    bin_edges = np.linspace(0.0, 1.0, number_of_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    binned_median = np.full(number_of_bins, np.nan)
    binned_std = np.full(number_of_bins, np.nan)

    bin_indices = np.digitize(phase, bin_edges) - 1

    # phase == 1 is equivalent to phase == 0.
    bin_indices[bin_indices == number_of_bins] = 0

    for bin_index in range(number_of_bins):
        mask = bin_indices == bin_index

        if np.any(mask):
            binned_median[bin_index] = np.nanmedian(values[mask])

            if np.count_nonzero(mask) >= 2:
                binned_std[bin_index] = np.nanstd(
                    values[mask],
                    ddof=1,
                )

    valid = np.isfinite(binned_median)

    return (
        bin_centers[valid],
        binned_median[valid],
        binned_std[valid],
    )


def plot_phase_folded_lightcurve(
    phase: np.ndarray,
    values: np.ndarray,
    column: str,
    period: float,
    object_name: str,
    intl_des: str,
    plot_two_cycles: bool = True,
    number_of_bins: int | None = None,
    output_path: Path | None = None,
) -> None:
    """Plot a phase-folded light curve."""

    sort_index = np.argsort(phase)
    sorted_phase = phase[sort_index]
    sorted_values = values[sort_index]

    if plot_two_cycles:
        plot_phase = np.concatenate(
            [sorted_phase, sorted_phase + 1.0]
        )
        plot_values = np.concatenate(
            [sorted_values, sorted_values]
        )
        x_limit = (0.0, 2.0)
    else:
        plot_phase = sorted_phase
        plot_values = sorted_values
        x_limit = (0.0, 1.0)

    plt.figure(figsize=(10, 5))

    plt.scatter(
        plot_phase,
        plot_values,
        s=10,
        alpha=0.45,
        label="Measurements",
    )

    if number_of_bins is not None:
        (
            bin_phase,
            bin_median,
            bin_std,
        ) = calculate_phase_binned_curve(
            phase,
            values,
            number_of_bins,
        )

        if plot_two_cycles:
            bin_phase_plot = np.concatenate(
                [bin_phase, bin_phase + 1.0]
            )
            bin_median_plot = np.concatenate(
                [bin_median, bin_median]
            )
            bin_std_plot = np.concatenate(
                [bin_std, bin_std]
            )
        else:
            bin_phase_plot = bin_phase
            bin_median_plot = bin_median
            bin_std_plot = bin_std

        plt.errorbar(
            bin_phase_plot,
            bin_median_plot,
            yerr=bin_std_plot,
            fmt="o-",
            markersize=4,
            linewidth=1.2,
            capsize=2,
            label=f"Median in {number_of_bins} phase bins",
        )

    plt.xlim(*x_limit)
    plt.xlabel("Phase")
    plt.ylabel(get_axis_label(column))
    plt.title(
        f"{object_name} ({intl_des}) phase-folded light curve\n"
        f"Period = {period:.6f} s"
    )
    plt.grid(alpha=0.3)

    if is_magnitude_column(column):
        plt.gca().invert_yaxis()

    plt.legend()
    plt.tight_layout()

    if output_path is not None:
        plt.savefig(output_path, dpi=200, bbox_inches="tight")

    plt.show()

def convert_to_json_value(value: Any) -> Any:
    """
    Convert NumPy scalar values into standard Python values
    that can be serialized by json.dump().
    """

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if not np.isfinite(value):
            return None
        return float(value)

    if isinstance(value, np.ndarray):
        return [
            convert_to_json_value(item)
            for item in value.tolist()
        ]

    return value


def save_phase_folded_lightcurve_json(
    kupt_data: dict[str, Any],
    output_path: Path,
    phase: np.ndarray,
    values: np.ndarray,
    original_ids: np.ndarray,
    value_column: str,
    period: float,
    period_source: str,
    phase_zero_time: float | None,
    estimated_period: float | None = None,
    number_of_bins: int | None = None,
) -> None:
    """
    Save a phase-folded light curve in a JSON structure based on
    the original KUPT-lightcurveJSON file.

    The output contains:
      - the original observation header
      - phase-folding settings
      - individual valid measurements sorted by phase
      - optional phase-binned median and standard deviation
    """

    phase = np.asarray(phase, dtype=float)
    values = np.asarray(values, dtype=float)
    original_ids = np.asarray(original_ids, dtype=int)

    if not (
        len(phase)
        == len(values)
        == len(original_ids)
    ):
        raise ValueError(
            "phase, values, and original_ids must have the same length."
        )

    if period <= 0:
        raise ValueError("The folding period must be positive.")

    if phase_zero_time is None:
        valid_time_values = [
            float(row["elapsed_seconds"])
            for row in kupt_data["data"]
            if row.get("elapsed_seconds") is not None
        ]

        if not valid_time_values:
            raise ValueError(
                "No valid elapsed_seconds values were found."
            )

        actual_phase_zero_time = min(valid_time_values)
    else:
        actual_phase_zero_time = float(phase_zero_time)

    # Create a lookup table from the original measurement ID.
    original_row_by_id: dict[int, dict[str, Any]] = {}

    for row in kupt_data["data"]:
        row_id = row.get("id")

        if row_id is None:
            continue

        try:
            original_row_by_id[int(row_id)] = row
        except (TypeError, ValueError):
            continue

    # Sort output measurements by phase.
    sort_index = np.argsort(phase)

    phase_sorted = phase[sort_index]
    values_sorted = values[sort_index]
    ids_sorted = original_ids[sort_index]

    phase_data: list[dict[str, Any]] = []

    for output_id, (
        original_id,
        phase_value,
        measured_value,
    ) in enumerate(
        zip(
            ids_sorted,
            phase_sorted,
            values_sorted,
        ),
        start=1,
    ):
        original_row = original_row_by_id.get(
            int(original_id),
            {},
        )

        output_row = {
            "id": output_id,
            "original_id": int(original_id),
            "phase": float(phase_value),
            "cycle_position_seconds": float(
                phase_value * period
            ),
            "elapsed_seconds": convert_to_json_value(
                original_row.get("elapsed_seconds")
            ),
            "time_start_utc": original_row.get(
                "time_start_utc"
            ),
            "time_mid_utc": original_row.get(
                "time_mid_utc"
            ),
            value_column: float(measured_value),
        }

        # Preserve related photometric quantities when available.
        for column_name in (
            "object_flux",
            "aperture_sum",
            "bkg_med_pix",
            "mag_in",
            "mag_app",
            "x_centroid",
            "y_centroid",
            "status",
        ):
            if column_name not in output_row:
                output_row[column_name] = (
                    convert_to_json_value(
                        original_row.get(column_name)
                    )
                )

        phase_data.append(output_row)

    output_header = deepcopy(
        kupt_data.get("header", {})
    )

    output_header["length"] = len(phase_data)

    output_header["phaseFoldSettings"] = {
        "valueColumn": value_column,
        "periodSeconds": float(period),
        "periodSource": period_source,
        "estimatedPeriodSeconds": (
            None
            if estimated_period is None
            else float(estimated_period)
        ),
        "phaseZeroElapsedSeconds": float(
            actual_phase_zero_time
        ),
        "phaseDefinition": (
            "((elapsed_seconds - "
            "phaseZeroElapsedSeconds) / "
            "periodSeconds) modulo 1"
        ),
        "phaseRange": [0.0, 1.0],
        "measurementsSortedBy": "phase",
        "numberOfValidMeasurements": len(phase_data),
    }

    output_header["phaseFoldEditor"] = {
        "appName": "Light Curve Viewer",
        "operation": "phase-folded light curve export",
        "createdUTC": datetime.now(
            timezone.utc
        ).isoformat(
            timespec="milliseconds"
        ).replace(
            "+00:00",
            "Z",
        ),
    }

    output_json: dict[str, Any] = {
        "header": output_header,
        "data": phase_data,
    }

    # Add phase-binned data when requested.
    if number_of_bins is not None:
        (
            bin_phase,
            bin_median,
            bin_std,
        ) = calculate_phase_binned_curve(
            phase=phase,
            values=values,
            number_of_bins=number_of_bins,
        )

        bin_edges = np.linspace(
            0.0,
            1.0,
            number_of_bins + 1,
        )

        phase_bin_indices = (
            np.digitize(phase, bin_edges) - 1
        )
        phase_bin_indices[
            phase_bin_indices == number_of_bins
        ] = 0

        binned_data: list[dict[str, Any]] = []

        valid_bin_number = 0

        for bin_index in range(number_of_bins):
            mask = phase_bin_indices == bin_index

            if not np.any(mask):
                continue

            median_value = np.nanmedian(values[mask])

            if not np.isfinite(median_value):
                continue

            valid_bin_number += 1

            if np.count_nonzero(mask) >= 2:
                std_value = np.nanstd(
                    values[mask],
                    ddof=1,
                )
            else:
                std_value = np.nan

            binned_data.append(
                {
                    "id": valid_bin_number,
                    "bin_index": bin_index,
                    "phase_start": float(
                        bin_edges[bin_index]
                    ),
                    "phase_center": float(
                        0.5
                        * (
                            bin_edges[bin_index]
                            + bin_edges[bin_index + 1]
                        )
                    ),
                    "phase_end": float(
                        bin_edges[bin_index + 1]
                    ),
                    "cycle_position_seconds": float(
                        0.5
                        * (
                            bin_edges[bin_index]
                            + bin_edges[bin_index + 1]
                        )
                        * period
                    ),
                    "number_of_measurements": int(
                        np.count_nonzero(mask)
                    ),
                    f"{value_column}_median": float(
                        median_value
                    ),
                    f"{value_column}_std": (
                        None
                        if not np.isfinite(std_value)
                        else float(std_value)
                    ),
                }
            )

        output_json["phaseBinnedData"] = {
            "numberOfBins": int(number_of_bins),
            "numberOfNonEmptyBins": len(binned_data),
            "statistic": "median",
            "scatter": "sample standard deviation",
            "data": binned_data,
        }

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            output_json,
            file,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )

    print()
    print(
        "Phase-folded light-curve JSON saved:"
    )
    print(output_path.resolve())

### Run

In [ ]:
# ============================================================
# Main analysis
# ============================================================

kupt_data = load_kupt_lightcurve_json(JSON_PATH)

display_observation_information(kupt_data)

object_name = str(
    get_nested(
        kupt_data["header"],
        "object",
        "objectName",
        default="Unknown object",
    )
)

intl_des = str(
    get_nested(
        kupt_data["header"],
        "object",
        "intlDES",
        default="Unknown INTLDES",
    )
)

if SAVE_FIGURES:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

    lightcurve_output = OUTPUT_DIRECTORY / f"{JSON_PATH.stem}_lightcurve.png"
    periodogram_output = OUTPUT_DIRECTORY / f"{JSON_PATH.stem}_periodogram.png"
    phase_output = OUTPUT_DIRECTORY / f"{JSON_PATH.stem}_phase_folded_lightcurve.png"
else:
    lightcurve_output = None
    periodogram_output = None
    phase_output = None

if SAVE_PHASE_FOLDED_JSON:
    OUTPUT_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True,
    )

    phase_json_output = (
        OUTPUT_DIRECTORY
        / f"{JSON_PATH.stem}_phase_folded.json"
    )
else:
    phase_json_output = None

# --------------------------------------------------------
# Extract and plot the displayed light curve
# --------------------------------------------------------

(
    plot_time,
    plot_values,
    plot_ids,
) = extract_lightcurve(
    kupt_data,
    PLOT_COLUMN,
)

print_section("Light-curve data summary")

print(f"Displayed column             : {PLOT_COLUMN}")
print(f"Valid displayed measurements: {len(plot_time)}")
print(f"First valid frame ID         : {plot_ids[0]}")
print(f"Last valid frame ID          : {plot_ids[-1]}")
print(f"Observation duration         : {np.ptp(plot_time):.6f} s")
print(f"Minimum value                : {np.min(plot_values):.6f}")
print(f"Maximum value                : {np.max(plot_values):.6f}")
print(f"Median value                 : {np.median(plot_values):.6f}")

if len(plot_time) >= 2:
    cadence = np.diff(plot_time)

    print(
        f"Median sampling interval     : "
        f"{np.median(cadence):.6f} s"
    )

plot_lightcurve(
    time_seconds=plot_time,
    values=plot_values,
    column=PLOT_COLUMN,
    object_name=object_name,
    intl_des=intl_des,
    output_path=lightcurve_output,
)

if PERIOD_ANAYSIS:
    # --------------------------------------------------------
    # Period analysis
    # --------------------------------------------------------

    (
        analysis_time,
        analysis_values,
        _,
    ) = extract_lightcurve(
        kupt_data,
        PERIOD_ANALYSIS_COLUMN,
    )

    print_section("Period analysis")

    print(
        f"Period-analysis column       : "
        f"{PERIOD_ANALYSIS_COLUMN}"
    )
    print(
        f"Valid analysis measurements  : "
        f"{len(analysis_time)}"
    )
    print(
        f"Search range                 : "
        f"{MIN_PERIOD:.6f}–{MAX_PERIOD:.6f} s"
    )

    (
        estimated_period,
        frequencies,
        power,
    ) = estimate_period_lomb_scargle(
        time_seconds=analysis_time,
        values=analysis_values,
        min_period=MIN_PERIOD,
        max_period=MAX_PERIOD,
        n_frequencies=N_FREQUENCIES,
    )

    print(
        f"Best Lomb-Scargle period     : "
        f"{estimated_period:.9f} s"
    )

    major_peaks = find_periodogram_peaks(
        frequencies,
        power,
        number_of_peaks=10,
    )

    print()
    print("Major Lomb-Scargle peaks:")
    print(f"{'Rank':>4s} {'Period [s]':>16s} {'Power':>16s}")

    for rank, (period, peak_power) in enumerate(
        major_peaks,
        start=1,
    ):
        print(
            f"{rank:4d} "
            f"{period:16.9f} "
            f"{peak_power:16.8f}"
        )

    plot_periodogram(
        frequencies=frequencies,
        power=power,
        best_period=estimated_period,
        object_name=object_name,
        intl_des=intl_des,
        output_path=periodogram_output,
    )

    # --------------------------------------------------------
    # Select folding period
    # --------------------------------------------------------

    if MANUAL_PERIOD is None:
        folding_period = estimated_period
        period_source = "Lomb-Scargle estimate"
    else:
        folding_period = float(MANUAL_PERIOD)
        period_source = "manual setting"

    print()
    print(f"Folding period               : {folding_period:.9f} s")
    print(f"Period source                : {period_source}")

    # --------------------------------------------------------
    # Phase-fold the displayed quantity
    # --------------------------------------------------------

    phase = calculate_phase(
        time_seconds=plot_time,
        period=folding_period,
        phase_zero_time=PHASE_ZERO_TIME,
    )

    plot_phase_folded_lightcurve(
        phase=phase,
        values=plot_values,
        column=PLOT_COLUMN,
        period=folding_period,
        object_name=object_name,
        intl_des=intl_des,
        plot_two_cycles=PLOT_TWO_PHASE_CYCLES,
        number_of_bins=N_PHASE_BINS,
        output_path=phase_output,
    )

    if SAVE_PHASE_FOLDED_JSON:
        save_phase_folded_lightcurve_json(
            kupt_data=kupt_data,
            output_path=phase_json_output,
            phase=phase,
            values=plot_values,
            original_ids=plot_ids,
            value_column=PLOT_COLUMN,
            period=folding_period,
            period_source=period_source,
            phase_zero_time=PHASE_ZERO_TIME,
            estimated_period=estimated_period,
            number_of_bins=(
                N_PHASE_BINS
                if SAVE_PHASE_BINNED_DATA
                else None
            ),
        )

### For Paper

In [ ]:
PAPER_FIGURE_SIZE_INCHES = (7.2, 8.4)
def plot_publication_lightcurve_figure(
    time_seconds: np.ndarray,
    values: np.ndarray,
    phase: np.ndarray,
    column: str,
    period: float,
    object_name: str,
    intl_des: str,
    number_of_bins: int,
    plot_two_cycles: bool,
    output_path: Path,
    figure_size_inches: tuple[float, float] = PAPER_FIGURE_SIZE_INCHES,
) -> None:
    """
    Create a publication figure containing two equally high panels.

    The upper panel shows the raw light curve. The lower panel shows only
    the phase-binned median curve; individual measurements and error bars
    are intentionally omitted.
    """

    if number_of_bins is None:
        raise ValueError(
            "N_PHASE_BINS must be an integer for the publication figure."
        )

    bin_phase, bin_median, _ = calculate_phase_binned_curve(
        phase=phase,
        values=values,
        number_of_bins=number_of_bins,
    )

    if plot_two_cycles:
        phase_to_plot = np.concatenate(
            [bin_phase, bin_phase + 1.0]
        )
        median_to_plot = np.concatenate(
            [bin_median, bin_median]
        )
        phase_limits = (0.0, 2.0)
    else:
        phase_to_plot = bin_phase
        median_to_plot = bin_median
        phase_limits = (0.0, 1.0)

    # height_ratios=(1, 1) guarantees equal panel heights.
    figure, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=figure_size_inches,
        gridspec_kw={
            "height_ratios": (1, 1),
            "hspace": 0.30,
        },
    )

    raw_axis, phase_axis = axes

    # Upper panel: raw light curve.
    raw_axis.scatter(
        time_seconds,
        values,
        s=8,
        alpha=0.80,
        linewidths=0,
    )
    raw_axis.set_xlabel("Elapsed time [s]")
    raw_axis.set_ylabel(get_axis_label(column))
    raw_axis.set_title(
        f"{object_name} ({intl_des})\n"
        "Raw light curve"
    )
    raw_axis.grid(alpha=0.3)

    # Lower panel: phase-binned median only.
    phase_axis.plot(
        phase_to_plot,
        median_to_plot,
        marker="o",
        markersize=2.5,
        linewidth=1.0,
    )
    phase_axis.set_xlim(*phase_limits)
    phase_axis.set_xlabel("Phase")
    phase_axis.set_ylabel(get_axis_label(column))
    phase_axis.set_title(
        f"Phase-folded light curve\n"
        f"Period = {period:.6f} s"
    )
    phase_axis.grid(alpha=0.3)

    if is_magnitude_column(column):
        raw_axis.invert_yaxis()
        phase_axis.invert_yaxis()

    # Use fixed margins rather than bbox_inches='tight' so that the PDF
    # retains the exact figure size specified above.
    figure.subplots_adjust(
        left=0.13,
        right=0.98,
        bottom=0.09,
        top=0.95,
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    figure.savefig(
        output_path,
        format="pdf",
        metadata={
            "Title": (
                f"{object_name} ({intl_des}) raw and phase-folded light curve"
            ),
            "Creator": "Light Curve Viewer",
        },
    )

    plt.show()
    plt.close(figure)

    print()
    print("Publication light-curve PDF saved:")
    print(output_path.resolve())

def plot_publication_lightcurve_figure_errorbar(
    time_seconds: np.ndarray,
    values: np.ndarray,
    phase: np.ndarray,
    column: str,
    period: float,
    object_name: str,
    intl_des: str,
    number_of_bins: int,
    plot_two_cycles: bool,
    output_path: Path,
    figure_size_inches: tuple[float, float] = PAPER_FIGURE_SIZE_INCHES,
) -> None:
    """
    Create a publication figure containing two equally high panels.

    The upper panel shows the raw light curve. The lower panel shows only
    the phase-binned median curve; individual measurements and error bars
    are intentionally omitted.
    """

    if number_of_bins is None:
        raise ValueError(
            "N_PHASE_BINS must be an integer for the publication figure."
        )

    bin_phase, bin_median, bin_std = calculate_phase_binned_curve(
        phase=phase,
        values=values,
        number_of_bins=number_of_bins,
    )

    sort_index = np.argsort(phase)
    sorted_phase = phase[sort_index]
    sorted_values = values[sort_index]

    if plot_two_cycles:
        measurement_phase = np.concatenate(
            [sorted_phase, sorted_phase + 1.0]
        )
        measurement_values = np.concatenate(
            [sorted_values, sorted_values]
        )

        phase_to_plot = np.concatenate(
            [bin_phase, bin_phase + 1.0]
        )
        median_to_plot = np.concatenate(
            [bin_median, bin_median]
        )
        std_to_plot = np.concatenate(
            [bin_std, bin_std]
        )

        phase_limits = (0.0, 2.0)

    else:
        measurement_phase = sorted_phase
        measurement_values = sorted_values

        phase_to_plot = bin_phase
        median_to_plot = bin_median
        std_to_plot = bin_std

        phase_limits = (0.0, 1.0)

    # height_ratios=(1, 1) guarantees equal panel heights.
    figure, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=figure_size_inches,
        gridspec_kw={
            "height_ratios": (1, 1),
            "hspace": 0.30,
        },
    )

    raw_axis, phase_axis = axes

    # Upper panel: raw light curve.
    raw_axis.scatter(
        time_seconds,
        values,
        s=8,
        alpha=0.80,
        linewidths=0,
    )
    raw_axis.set_xlabel("Elapsed time [s]")
    raw_axis.set_ylabel(get_axis_label(column))
    raw_axis.set_title(
        f"{object_name} ({intl_des})\n"
        "Raw light curve"
    )
    raw_axis.grid(alpha=0.3)

    # Lower panel: phase-binned median only.
    # Lower panel: individual measurements in gray.
    phase_axis.scatter(
        measurement_phase,
        measurement_values,
        s=5,
        color="0.70",
        alpha=0.45,
        linewidths=0,
        label="Measurements",
        zorder=1,
    )

    # Phase-binned median and standard-deviation error bars in gray.
    phase_axis.errorbar(
        phase_to_plot,
        median_to_plot,
        yerr=std_to_plot,
        fmt="none",
        ecolor="0.55",
        elinewidth=0.7,
        capsize=1.5,
        alpha=0.8,
        zorder=2,
    )

    # Median curve.
    phase_axis.plot(
        phase_to_plot,
        median_to_plot,
        marker="o",
        markersize=2.5,
        linewidth=1.1,
        label=f"Median in {number_of_bins} phase bins",
        zorder=3,
    )

    phase_axis.legend(
        loc="best",
        fontsize=8,
    )
    phase_axis.set_xlim(*phase_limits)
    phase_axis.set_xlabel("Phase")
    phase_axis.set_ylabel(get_axis_label(column))
    phase_axis.set_ylim([12,22])
    phase_axis.set_title(
        f"Phase-folded light curve\n"
        f"Period = {period:.6f} s"
    )
    phase_axis.grid(alpha=0.3)

    if is_magnitude_column(column):
        raw_axis.invert_yaxis()
        phase_axis.invert_yaxis()

    # Use fixed margins rather than bbox_inches='tight' so that the PDF
    # retains the exact figure size specified above.
    figure.subplots_adjust(
        left=0.13,
        right=0.98,
        bottom=0.09,
        top=0.95,
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    figure.savefig(
        output_path,
        format="pdf",
        metadata={
            "Title": (
                f"{object_name} ({intl_des}) raw and phase-folded light curve"
            ),
            "Creator": "Light Curve Viewer",
        },
    )

    plt.show()
    plt.close(figure)

    print()
    print("Publication light-curve PDF saved:")
    print(output_path.resolve())

if PERIOD_ANAYSIS:
    publication_figure_output = (
        OUTPUT_DIRECTORY
        / f"{JSON_PATH.stem}_raw_and_phase_folded_lightcurve.pdf"
    )

    plot_publication_lightcurve_figure_errorbar(
        time_seconds=plot_time,
        values=plot_values,
        phase=phase,
        column=PLOT_COLUMN,
        period=folding_period,
        object_name=object_name,
        intl_des=intl_des,
        number_of_bins=N_PHASE_BINS,
        plot_two_cycles=PLOT_TWO_PHASE_CYCLES,
        output_path=publication_figure_output,
    )
